In [1]:
# ============================================================
# BDM Projekt SS26 — Data Quality & Anomaly Detection
# Datensatz: Fraud Detection Dataset (Kaggle)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# Datensatz laden
df = pd.read_csv('Fraud Detection Dataset.csv')

# Erste Übersicht
print("=== SHAPE ===")

print(f"Zeilen: {df.shape[0]}, Spalten: {df.shape[1]}")

print("\n=== SPALTEN & DATENTYPEN ===")
print(df.dtypes)

print("\n=== ERSTE 5 ZEILEN ===")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Fraud Detection Dataset.csv'

In [ ]:
# ============================================================
# BLOCK 2 — Datenprofiling
# ============================================================

print("=== FEHLENDE WERTE ===")
missing = df.isnull().sum()
missing_pct = df.isnull().sum() / len(df) * 100
missing_df = pd.DataFrame({
    'Fehlend (Anzahl)': missing,
    'Fehlend (%)': round(missing_pct, 2)
})
print(missing_df[missing_df['Fehlend (Anzahl)'] > 0])

print("\n=== DUPLIKATE ===")
print(f"Doppelte Transaction_IDs: {df['Transaction_ID'].duplicated().sum()}")
print(f"Komplette Zeilen-Duplikate: {df.duplicated().sum()}")

print("\n=== VERDÄCHTIGE WERTE ===")
print(f"'Unknown Device' in Device_Used: {(df['Device_Used'] == 'Unknown Device').sum()}")
print(f"'Invalid Method' in Payment_Method: {(df['Payment_Method'] == 'Invalid Method').sum()}")

print("\n=== STATISTIK ===")
df.describe()

In [ ]:
# ============================================================
# BLOCK 3 — Visualisierung
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Datenqualität — Fraud Detection Dataset', fontsize=16, fontweight='bold')

# Plot 1: Fehlende Werte ---
missing_pct = df.isnull().sum() / len(df) * 100
missing_pct = missing_pct[missing_pct > 0]

axes[0].barh(missing_pct.index, missing_pct.values, color='#E74C3C')
axes[0].set_title('Fehlende Werte (%)', fontweight='bold')
axes[0].set_xlabel('Prozent (%)')
for i, v in enumerate(missing_pct.values):
    axes[0].text(v + 0.05, i, f'{v:.2f}%', va='center')

# Plot 2: Duplikate ---
dup_labels = ['Eindeutige IDs', 'Doppelte IDs']
dup_values = [len(df) - df['Transaction_ID'].duplicated().sum(), 
              df['Transaction_ID'].duplicated().sum()]

axes[1].bar(dup_labels, dup_values, color=['#2ECC71', '#E74C3C'])
axes[1].set_title('Duplikate — Transaction_ID', fontweight='bold')
axes[1].set_ylabel('Anzahl')
for i, v in enumerate(dup_values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Plot 3: Verdächtige Werte ---
sus_labels = ['Unknown Device', 'Invalid Method']
sus_values = [
    (df['Device_Used'] == 'Unknown Device').sum(),
    (df['Payment_Method'] == 'Invalid Method').sum()
]

axes[2].bar(sus_labels, sus_values, color='#E67E22')
axes[2].set_title('Verdächtige Werte', fontweight='bold')
axes[2].set_ylabel('Anzahl')
for i, v in enumerate(sus_values):
    axes[2].text(i, v + 10, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('datenqualitaet.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# BLOCK 3b — Datentypen
# ============================================================

print("=== DATENTYPEN ===")
print(df.dtypes)

print("\n=== PROBLEM: Spalten die numerisch sein sollten ===")
print(f"Transaction_Amount Typ: {df['Transaction_Amount'].dtype} ")
print(f"Time_of_Transaction Typ: {df['Time_of_Transaction'].dtype} ")
print(f"Fraudulent Typ: {df['Fraudulent'].dtype} ")
print(f"Transaction_ID Typ: {df['Transaction_ID'].dtype} — Text, nicht Zahl (normal)")
print(f"Device_Used Typ: {df['Device_Used'].dtype} — Text (normal)")
print(f"Payment_Method Typ: {df['Payment_Method'].dtype} — Text (normal)")

In [ ]:
# ============================================================
# BLOCK 4 — Ausreißer (Outliers)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Ausreißer — Numerische Spalten', fontsize=16, fontweight='bold')

# Plot 1: Transaction_Amount ---
axes[0].boxplot(df['Transaction_Amount'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#3498DB', alpha=0.7))
axes[0].set_title('Transaction_Amount', fontweight='bold')
axes[0].set_ylabel('Betrag ($)')

# Plot 2: Account_Age ---
axes[1].boxplot(df['Account_Age'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#2ECC71', alpha=0.7))
axes[1].set_title('Account_Age', fontweight='bold')
axes[1].set_ylabel('Alter (Tage)')

# Plot 3: Number_of_Transactions_Last_24H ---
axes[2].boxplot(df['Number_of_Transactions_Last_24H'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#E67E22', alpha=0.7))
axes[2].set_title('Transaktionen letzte 24H', fontweight='bold')
axes[2].set_ylabel('Anzahl')

plt.tight_layout()
plt.savefig('ausreisser.png', dpi=150, bbox_inches='tight')
plt.show()

# Ausreißer berechnen mit IQR-Methode
print("=== AUSREISSER (IQR-Methode) ===")
for col in ['Transaction_Amount', 'Account_Age', 'Number_of_Transactions_Last_24H']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    untere_grenze = Q1 - 1.5 * IQR
    obere_grenze = Q3 + 1.5 * IQR
    ausreisser = df[(df[col] < untere_grenze) | (df[col] > obere_grenze)]
    print(f"\n{col}:")
    print(f"  Untere Grenze: {untere_grenze:.2f}")
    print(f"  Obere Grenze: {obere_grenze:.2f}")
    print(f"  Anzahl Ausreißer: {len(ausreisser)}")

In [ ]:
# ============================================================
# BLOCK 5 — Quality-Checks (5 Regeln)
# ============================================================

print("=== QUALITY-CHECKS ===\n")

# QC1: Transaction_Amount muss positiv sein
qc1 = df[df['Transaction_Amount'] <= 0]
print(f"QC1 — Transaction_Amount <= 0: {len(qc1)} Verstöße")

# QC2: Keine doppelten Transaction_IDs
qc2 = df['Transaction_ID'].duplicated().sum()
print(f"QC2 — Doppelte Transaction_IDs: {qc2} Verstöße")

# QC3: Time_of_Transaction muss zwischen 0 und 23 liegen
qc3 = df[(df['Time_of_Transaction'] < 0) | (df['Time_of_Transaction'] > 23)]
print(f"QC3 — Time_of_Transaction außerhalb 0-23: {len(qc3)} Verstöße")

# QC4: Fraudulent darf nur 0 oder 1 sein
qc4 = df[~df['Fraudulent'].isin([0, 1])]
print(f"QC4 — Fraudulent nicht 0/1: {len(qc4)} Verstöße")

# QC5: Payment_Method und Device_Used dürfen nicht "Invalid"/"Unknown" sein
qc5_a = (df['Payment_Method'] == 'Invalid Method').sum()
qc5_b = (df['Device_Used'] == 'Unknown Device').sum()
print(f"QC5 — Invalid Method: {qc5_a} Verstöße")
print(f"QC5 — Unknown Device: {qc5_b} Verstöße")

# Zusammenfassung
print("\n=== ZUSAMMENFASSUNG ===")
total_checks = 5
failed_checks = sum([len(qc1) > 0, qc2 > 0, len(qc3) > 0, len(qc4) > 0, (qc5_a + qc5_b) > 0])
print(f"Checks insgesamt: {total_checks}")
print(f"Checks mit Verstößen: {failed_checks}")
print(f"Checks ohne Verstöße: {total_checks - failed_checks}")

In [ ]:
# ============================================================
# BLOCK 6 — Isolation Forest
# ============================================================

from sklearn.ensemble import IsolationForest

# Vorbereitung: nur numerische Spalten, fehlende Werte entfernen
features = ['Transaction_Amount', 'Time_of_Transaction', 
             'Previous_Fraudulent_Transactions', 'Account_Age', 
             'Number_of_Transactions_Last_24H']

df_model = df.dropna(subset=features).copy()
print(f"Zeilen vor dropna: {len(df)}")
print(f"Zeilen nach dropna: {len(df_model)}")

X = df_model[features]

# Isolation Forest trainieren
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_model['anomaly'] = iso_forest.fit_predict(X)

# -1 = Anomalie, 1 = normal
df_model['anomaly_label'] = df_model['anomaly'].map({1: 'Normal', -1: 'Anomalie'})

print("\n=== ERGEBNIS ===")
print(df_model['anomaly_label'].value_counts())

In [ ]:
# ============================================================2
# BLOCK 7 — Vergleich: Anomalien vs. echte Fraud-Labels
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report

# Anomalie (-1) → 1, Normal (1) → 0  (gleiche Kodierung wie Fraudulent)
df_model['anomaly_binary'] = df_model['anomaly'].map({-1: 1, 1: 0})

# Confusion Matrix
cm = confusion_matrix(df_model['Fraudulent'], df_model['anomaly_binary'])
print("=== CONFUSION MATRIX ===")
print("                 Vorhergesagt: Normal | Vorhergesagt: Anomalie")
print(f"Tatsächlich Normal:    {cm[0][0]:>8}        |  {cm[0][1]:>8}")
print(f"Tatsächlich Fraud:     {cm[1][0]:>8}        |  {cm[1][1]:>8}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(df_model['Fraudulent'], df_model['anomaly_binary'], 
                              target_names=['Normal', 'Fraud']))

# Wie viel Prozent der echten Fraud-Fälle wurden gefunden?
total_fraud = df_model['Fraudulent'].sum()
found_fraud = cm[1][1]
print(f"\nEchte Fraud-Fälle insgesamt: {total_fraud}")
print(f"Davon von Isolation Forest als Anomalie erkannt: {found_fraud}")
print(f"Erkennungsrate: {found_fraud/total_fraud*100:.1f}%")

In [ ]:
# ============================================================
# BLOCK 8 — Visualisierung: Anomalien vs. Fraud
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Isolation Forest — Anomalien vs. echte Fraud-Fälle', fontsize=14, fontweight='bold')

# Plot 1: Transaction_Amount nach Anomalie-Status ---
axes[0].scatter(df_model[df_model['anomaly_label']=='Normal']['Transaction_Amount'],
                df_model[df_model['anomaly_label']=='Normal']['Account_Age'],
                alpha=0.3, s=10, color='#3498DB', label='Normal')
axes[0].scatter(df_model[df_model['anomaly_label']=='Anomalie']['Transaction_Amount'],
                df_model[df_model['anomaly_label']=='Anomalie']['Account_Age'],
                alpha=0.6, s=15, color='#E74C3C', label='Anomalie')
axes[0].set_title('Isolation Forest Anomalien', fontweight='bold')
axes[0].set_xlabel('Transaction_Amount ($)')
axes[0].set_ylabel('Account_Age (Tage)')
axes[0].legend()

# Plot 2: Confusion Matrix als Heatmap ---
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Anomalie'],
            yticklabels=['Normal', 'Fraud'])
axes[1].set_title('Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Von Isolation Forest vorhergesagt')
axes[1].set_ylabel('Tatsächlich (Fraud-Label)')

plt.tight_layout()
plt.savefig('isolation_forest_ergebnis.png', dpi=150, bbox_inches='tight')
plt.show()
